# Research 23 — Blend balanced weights with VIP × engagement

**Hypothesis:** the balanced-weight LightGBM and VIP × engagement LightGBM may correct different baseline errors. Compare their fixed 80/10/10 probability mixture (current/weights/interaction) with the current reference and both 80/20 single-candidate mixtures. All component predictions are aligned 3-fold OOF probabilities from train; this is one fixed synthesis, not a weight search. Test is not loaded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert np.all(sums > 0), filename
    return probs / sums


p_base = load_probs("segment-oof.csv")
p_balanced = load_probs("segment-lightgbm-balanced-weights-oof.csv")
p_interaction = load_probs("segment-vip-engagement-oof.csv")
candidates = {
    "Current reference": p_base,
    "80/20 balanced-weight": 0.8 * p_base + 0.2 * p_balanced,
    "80/20 VIP × engagement": 0.8 * p_base + 0.2 * p_interaction,
    "80/10/10 balanced + VIP × engagement": 0.8 * p_base + 0.1 * p_balanced + 0.1 * p_interaction,
}
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(np.zeros(len(y)), y))
rows = []
for name, probs in candidates.items():
    pred = probs.argmax(axis=1)
    row = {
        "candidate": name,
        "macro_f1": f1_score(y, pred, average="macro"),
        "weighted_f1": f1_score(y, pred, average="weighted"),
    }
    for fold, (_, valid_idx) in enumerate(folds, start=1):
        row[f"fold_{fold}_macro_f1"] = f1_score(y[valid_idx], pred[valid_idx], average="macro")
    rows.append(row)
results = pd.DataFrame(rows)
base_folds = results.loc[
    results.candidate == "Current reference", [f"fold_{fold}_macro_f1" for fold in range(1, 4)]
].iloc[0]
for fold in range(1, 4):
    results[f"fold_{fold}_delta"] = (
        results[f"fold_{fold}_macro_f1"] - base_folds[f"fold_{fold}_macro_f1"]
    )
display(results.round(6))
results.to_csv(ROOT / "assets/segment-balanced-vipengagement-threeway.csv", index=False)
pd.DataFrame(
    candidates["80/10/10 balanced + VIP × engagement"], columns=[f"prob_{i}" for i in range(4)]
).assign(segment=y).to_csv(
    ROOT / "predictions/segment-balanced-vipengagement-threeway-oof.csv", index=False
)
print("Fixed probability synthesis complete; train OOF only, Test not loaded.")

,candidate,macro_f1,weighted_f1,fold_1_macro_f1,fold_2_macro_f1,fold_3_macro_f1,fold_1_delta,fold_2_delta,fold_3_delta
0,Current reference,0.750895,0.764683,0.749445,0.749821,0.753390,0.000000,0.000000,0.000000
1,80/20 balanced-weight,0.751550,0.764986,0.749726,0.750842,0.754051,0.000280,0.001021,0.000661
2,80/20 VIP × engagement,0.751447,0.765293,0.748976,0.751254,0.754079,-0.000470,0.001434,0.000689
3,80/10/10 balanced + VIP × engagement,0.751406,0.765046,0.749236,0.751366,0.753585,-0.000209,0.001545,0.000195


Fixed probability synthesis complete; train OOF only, Test not loaded.


## Decision rule

Keep the three-way recipe as a screening candidate only if it beats both 80/20 blends without a material weighted-F1 regression and produces gains across the existing fold slices. The comparison reuses OOF training labels and does not estimate model-selection uncertainty.